In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]   # COLEPV1
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:

from colep_ai.database.qdrant_page_client import get_qdrant_client,ensure_page_collection
from colep_ai.indexing.embedder import get_openai_client
from colep_ai.retrieval.page_retrieval import retrieve


In [3]:
qdrant_client=get_qdrant_client()
openai_client=get_openai_client()

In [4]:
ensure_page_collection(qdrant_client)

2026-07-10 14:49:05 | INFO | httpx._client | HTTP Request: GET https://6ec1c2a5-4b08-472c-9229-91a89d3abd5e.us-west-1-0.aws.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
2026-07-10 14:49:05 | INFO | httpx._client | HTTP Request: GET https://6ec1c2a5-4b08-472c-9229-91a89d3abd5e.us-west-1-0.aws.cloud.qdrant.io:6333/collections/colep_page_based_chunks/exists "HTTP/1.1 200 OK"
2026-07-10 14:49:05 | INFO | colep_ai.database.qdrant_page_client | Collection already exists:colep_page_based_chunks


In [5]:
qdrant_client.get_collection("colep_page_based_chunks")

2026-07-10 14:49:06 | INFO | httpx._client | HTTP Request: GET https://6ec1c2a5-4b08-472c-9229-91a89d3abd5e.us-west-1-0.aws.cloud.qdrant.io:6333/collections/colep_page_based_chunks "HTTP/1.1 200 OK"


CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=5, segments_count=2, config=CollectionConfig(params=CollectionParams(vectors={'image_desc': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), 'text_en': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), 'text_pt': VectorParams(size=3072, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None)}, shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_sca

In [6]:
query = "How do I turn line O001.O059.1 on and off?"
chunks = retrieve(query, top_k=5,qdrant_client=get_qdrant_client(),openai_client=get_openai_client())

2026-07-10 14:49:06 | INFO | colep_ai.retrieval.page_retrieval | Detected language: 'english' (0.89) for query: 'How do I turn line O001.O059.1 on and off?'
2026-07-10 14:49:06 | INFO | colep_ai.retrieval.page_retrieval | Searching vector='text_en' top_k=5
2026-07-10 14:49:06 | INFO | httpx._client | HTTP Request: GET https://6ec1c2a5-4b08-472c-9229-91a89d3abd5e.us-west-1-0.aws.cloud.qdrant.io:6333 "HTTP/1.1 200 OK"
2026-07-10 14:49:09 | INFO | httpx._client | HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2026-07-10 14:49:10 | INFO | httpx._client | HTTP Request: POST https://6ec1c2a5-4b08-472c-9229-91a89d3abd5e.us-west-1-0.aws.cloud.qdrant.io:6333/collections/colep_page_based_chunks/points/query?timeout=5 "HTTP/1.1 200 OK"
2026-07-10 14:49:11 | INFO | httpx._client | HTTP Request: POST https://6ec1c2a5-4b08-472c-9229-91a89d3abd5e.us-west-1-0.aws.cloud.qdrant.io:6333/collections/colep_page_based_chunks/points/query?timeout=5 "HTTP/1.1 200 OK"
2026-07-10 14:4

In [7]:
chunks

RetrievalResponse(results=[{'payload': {'source_file': 'e1', 'page_number': 1, 'document_title': 'Ligar e desligar a linha - O01.O059.1', 'document_code': 'O01.O059.1', 'entries': [{'entry_id': 'row_1', 'parent_id': '', 'entry_text': 'Nº Edição: 1 | Data de revisão: 10/31/2025 | Descrição da Alteração: Edição Inicial por alteração do documento O01.I519.1', 'entry_text_en': 'Edition No.: 1 | Revision Date: 10/31/2025 | Description of Change: Initial Edition due to change of document O01.I519.1', 'fields': {'Nº Edição': '1', 'Data de revisão': '10/31/2025', 'Descrição da Alteração': 'Edição Inicial por alteração do documento O01.I519.1'}, 'image_ids': [], 'image_description': '', 'is_combined': False, 'combined_image': None}, {'entry_id': 'row_2', 'parent_id': '', 'entry_text': 'Nº Operação: 1 | Páginas: 2-3 | Título: Ligar a linha | Responsabilidade: Operador', 'entry_text_en': 'Operation No.: 1 | Pages: 2-3 | Title: Turn on the line | Responsibility: Operator', 'fields': {'Nº Operação'

In [8]:
# for i in chunks:
#     print(i)
#     print(i["payload"]["entries"][0]["entry_id"])
    
        #   print(r["score"], r["entry_text_en"],r["image_ids"])

In [9]:
from colep_ai.generation.answer_builder import generate_from_retrieval
import anthropic
client = anthropic.Anthropic()


In [10]:
r1=generate_from_retrieval(claude_client=client,query=query,retrieval_response=chunks)

2026-07-10 14:49:11 | INFO | colep_ai.generation.answer_builder | Generating answer | model=claude-sonnet-4-6 | answer_language=English
2026-07-10 14:49:26 | INFO | httpx._client | HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 OK"


In [11]:
r1

{'answer': '# Turning Line O01.O059.1 On and Off\n\n---\n\n## OPERATION 1 — Turn On the Line\n\n1. **Turn on the general power of the line** at the electrical panel (labeled "GERAL").\n🖼️[page 2 | entry 1]\n\n2. **Turn on the general power of the welding machine** by pressing the button indicated on the control box labeled "Corte Geral Corrente Elétrica Ar comprimido".\n🖼️[page 2 | entry 2]\n\n3. **Turn on the welding machine** by pressing the 4 green buttons on the control panel.\n🖼️[page 2 | entry 3]\n\n4. **Turn on the exterior varnish system** by pressing the "ON" button on the machine control panel.\n🖼️[page 2 | entry 4]\n\n5. **If the production order indicates sheet with interior varnish**, turn on the interior powder varnish machine using the red rotary switch.\n🖼️[page 2 | entry 5]\n\n6. **Open the gas tap** indicated on the yellow gas pipe assembly.\n🖼️[page 2 | entry 6]\n\n7. **Open the water taps** (two valves) indicated on the pipe installations.\n🖼️[page 2 | entry 7]\n\n8

In [12]:
c=r1["citations"]
c

[{'marker': '[page 2 | entry 1]',
  'image_ref': 'page_2_9347',
  'image_description': "The image shows an electrical panel labeled 'GERAL' mounted on a wall. The panel contains circuit breakers or switches. No specific red-marked region is distinctly visible in this thumbnail, but the panel is the referenced general electrical board for the line. The label 'GERAL' confirms this is the main electrical panel referenced in step 1."},
 {'marker': '[page 2 | entry 2]',
  'image_ref': 'page_2_87dc',
  'image_description': "The image shows a control box labeled 'Corte Geral Corrente Elétrica Ar comprimido' with several buttons and switches. A red circle highlights a specific button (appears to be a green start button) on the panel face. This red marking guides the operator to the exact button to press to turn on the general power of the welding machine, as instructed in step 2."},
 {'marker': '[page 2 | entry 3]',
  'image_ref': 'page_2_7c9f',
  'image_description': "The image shows the weld

In [13]:
ans=r1["answer"]

In [14]:
ans

'# Turning Line O01.O059.1 On and Off\n\n---\n\n## OPERATION 1 — Turn On the Line\n\n1. **Turn on the general power of the line** at the electrical panel (labeled "GERAL").\n🖼️[page 2 | entry 1]\n\n2. **Turn on the general power of the welding machine** by pressing the button indicated on the control box labeled "Corte Geral Corrente Elétrica Ar comprimido".\n🖼️[page 2 | entry 2]\n\n3. **Turn on the welding machine** by pressing the 4 green buttons on the control panel.\n🖼️[page 2 | entry 3]\n\n4. **Turn on the exterior varnish system** by pressing the "ON" button on the machine control panel.\n🖼️[page 2 | entry 4]\n\n5. **If the production order indicates sheet with interior varnish**, turn on the interior powder varnish machine using the red rotary switch.\n🖼️[page 2 | entry 5]\n\n6. **Open the gas tap** indicated on the yellow gas pipe assembly.\n🖼️[page 2 | entry 6]\n\n7. **Open the water taps** (two valves) indicated on the pipe installations.\n🖼️[page 2 | entry 7]\n\n8. **Turn on